Combinatorial Routing and Constraint Satisfaction Neural Solvers

Neural heuristics and continuous relaxations for NP-hard discrete optimization:
1. TSP Closed-Loop Tour Length: Differentiable Euclidean metric accumulation over permutations.
2. Pointer Networks: Sequence attention with dynamic visited masking for routing decisions.
3. GNN Edge Heatmaps: Pairwise edge classification for non-autoregressive tour prediction.
4. Continuous Max-Cut Relaxation: Differentiable energy Hamiltonians via hyperbolic tangent spins.

## 2. Mathematical Formulations & Theoretical Foundations

- **TSP Tour Cost**:
  $$\text{Cost}(\pi) = \sum_{i=1}^{N-1} \|x_{\pi_i} - x_{\pi_{i+1}}\|_2 + \|x_{\pi_N} - x_{\pi_1}\|_2$$
- **Pointer Network Masked Attention**:
  $$u_i = v^T \tanh(W_1 e_i + W_2 d), \quad u'_i = \begin{cases} u_i & \text{if unvisited} \\ -\infty & \text{if visited} \end{cases}, \quad p = \operatorname{softmax}(u')$$
- **GNN Edge Heatmap**:
  $$A_{i, j} = \sigma\left(W_{\text{edge}} [h_i, h_j, d_{ij}] + b_{\text{edge}}\right)$$
- **Relaxed Max-Cut Cost**:
  $$\text{Cut}(\sigma) = \frac{1}{4} \sum_{i, j} W_{ij} \left(1 - \tanh(z_i) \tanh(z_j)\right)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "07_domain_tasks" / "12_combinatorial_routing_and_constraint_satisfaction_neural_solvers.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. TSP Closed-Loop Tour Cost
def tsp_tour_cost(coords: torch.Tensor, tour: torch.Tensor) -> torch.Tensor:
    tour_expanded = tour.unsqueeze(-1).expand(-1, -1, 2)
    ordered_coords = torch.gather(coords, 1, tour_expanded)
    next_coords = torch.roll(ordered_coords, shifts=-1, dims=1)
    segment_lengths = torch.sqrt(torch.sum((ordered_coords - next_coords) ** 2, dim=-1) + 1e-12)
    return torch.sum(segment_lengths, dim=-1)

cost_calc = tsp_tour_cost(golden["coords"], golden["tour"])
diff_cost = torch.max(torch.abs(cost_calc - golden["tsp_cost_ref"])).item()
print(f"TSP Tour Cost max abs error: {diff_cost:.2e}")
assert diff_cost < 1e-5

In [ ]:
# 2. Pointer Network Step
def pointer_network_step(
    node_embeds: torch.Tensor,
    decoder_state: torch.Tensor,
    visited_mask: torch.Tensor,
    w1: torch.Tensor,
    w2: torch.Tensor,
    v: torch.Tensor,
) -> torch.Tensor:
    e1 = node_embeds @ w1
    e2 = (decoder_state @ w2).unsqueeze(1)
    scores = torch.tanh(e1 + e2) @ v
    masked_scores = scores.masked_fill(visited_mask.bool(), -1e9)
    return F.softmax(masked_scores, dim=-1)

ptr_calc = pointer_network_step(
    golden["node_embeds"],
    golden["decoder_state"],
    golden["visited_mask"],
    golden["w1"],
    golden["w2"],
    golden["v"],
)
diff_ptr = torch.max(torch.abs(ptr_calc - golden["ptr_probs_ref"])).item()
print(f"Pointer Network max abs error: {diff_ptr:.2e}")
assert diff_ptr < 1e-5

In [ ]:
# 3. GNN Edge Classification Heatmap
def gnn_edge_heatmap(
    h_nodes: torch.Tensor,
    edge_dists: torch.Tensor,
    w_edge: torch.Tensor,
    b_edge: torch.Tensor,
) -> torch.Tensor:
    b, n, d = h_nodes.shape
    hi = h_nodes.unsqueeze(2).expand(-1, -1, n, -1)
    hj = h_nodes.unsqueeze(1).expand(-1, n, -1, -1)
    dist_exp = edge_dists.unsqueeze(-1)
    edge_feats = torch.cat([hi, hj, dist_exp], dim=-1)
    logits = edge_feats @ w_edge + b_edge
    return torch.sigmoid(logits.squeeze(-1))

heat_calc = gnn_edge_heatmap(golden["h_nodes"], golden["edge_dists"], golden["w_edge"], golden["b_edge"])
diff_heat = torch.max(torch.abs(heat_calc - golden["heatmap_ref"])).item()
print(f"GNN Edge Heatmap max abs error: {diff_heat:.2e}")
assert diff_heat < 1e-5

In [ ]:
# 4. Continuous Relaxed Max-Cut Cost & Summary Table
def relaxed_maxcut_cost(z_spins: torch.Tensor, w_adj: torch.Tensor) -> torch.Tensor:
    sigma = torch.tanh(z_spins)
    spin_outer = sigma.unsqueeze(2) * sigma.unsqueeze(1)
    cut_matrix = 0.25 * w_adj * (1.0 - spin_outer)
    return torch.sum(cut_matrix, dim=(-2, -1))

maxcut_calc = relaxed_maxcut_cost(golden["z_spins"], golden["w_adj"])
diff_maxcut = torch.max(torch.abs(maxcut_calc - golden["maxcut_cost_ref"])).item()

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'TSP Tour Cost':<32} | {diff_cost:<15.2e} | PASSED")
print(f"{'Pointer Network Attention':<32} | {diff_ptr:<15.2e} | PASSED")
print(f"{'GNN Edge Heatmap':<32} | {diff_heat:<15.2e} | PASSED")
print(f"{'Relaxed Max-Cut Cost':<32} | {diff_maxcut:<15.2e} | PASSED")
print("=" * 65)
assert diff_maxcut < 1e-5

## 8. Summary, Invariants & Diagnostic Checklist

- [x] TSP tour evaluations provide differentiable objective signals for reinforcement learning and local search.
- [x] Pointer Networks dynamically select variable-length sequence elements while enforcing permutation constraints.
- [x] Graph Neural Networks generate global edge probability matrices, sparsifying combinatorial search spaces.
- [x] Continuous spin relaxation maps discrete NP-hard graph cuts into smooth, gradient-friendly optimization landscapes.